In [1]:
import pandas as pd
from bertopic import BERTopic
from sklearn.datasets import fetch_20newsgroups
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

import openai
from bertopic.representation import OpenAI

In [2]:
df_news = pd.read_parquet('./Data/news_date_-1_1.parquet')
df_news

,Date,Caption,Content
0,1984-12-31,Manager's Journal: Boosting Creativity and Dis...,An information-systems manager has assembled i...
1,1984-12-31,National Color Laboratories Inc.,"ROSELLE, N.J. -- National Color Laboratories I..."
2,1984-12-31,Massachusetts Agency Approves Textron Inc.'s P...,BOSTON -- The Massachusetts Division of Insura...
3,1984-12-31,Michigan General Corp.,Michigan General Corp. (Dallas) -- Bernard E. ...
4,1984-12-31,"Midway Air Shelves Helicopter Service Idea, Se...",WASHINGTON -- Midway Airlines dropped immediat...
...,...,...,...
98995,2020-03-25,Life & Arts: Hip-Hop in the Jazz Club,"Kassa Overall calls himself ""a backpack jazz p..."
98996,2020-03-25,Expose First Responders to the Coronavirus,Among many grim scenarios coming out of the co...
98997,2020-03-25,The Happy Few Are the Cured,The giddiest among us soon will be those who t...
98998,2020-03-25,Politics & Ideas: Covid-19 Relief Should Bar S...,As Covid-19 spreads and the economy falls off ...


In [3]:
docs = df_news['Content'].tolist()
timestamps = df_news['Date'].tolist()

In [4]:
# 1. 选择一个预训练的嵌入模型
# 'all-MiniLM-L6-v2' 是一个速度和性能都很均衡的通用模型
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# 2. 初始化 BERTopic 模型
# verbose=True 会在训练时打印进度信息
# embedding_model 参数指定了我们选择的嵌入模型
topic_model = BERTopic(embedding_model=embedding_model, 
                       min_topic_size=230,
                       verbose=True)

# 3. 训练模型并提取主题
# fit_transform 会返回每个文档对应的主题编号
# 对于有时间戳的数据，可以直接传入 timestamps 参数
topics, probs = topic_model.fit_transform(docs)

2025-06-28 02:10:03,197 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/3094 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [5]:
# 获取主题的概览信息
# Count: 该主题下的文档数量
# Name: BERTopic自动生成的主题名称（由前4个关键词和下划线组成）
# Representation: 代表该主题的关键词列表
topic_info = topic_model.get_topic_info()
print("Topic Info:")
print(topic_info)

# 主题-1是离群点，即模型认为不属于任何特定主题的文档。
# 我们可以看到每个主题的大小，方便我们关注那些规模较大的主题。

Topic Info:
    Topic  Count                                    Name  \
0      -1  43114                         -1_the_to_of_in   
1       0   4378                          0_the_to_in_of   
2       1   3294                         1_tax_the_to_mr   
3       2   3218               2_airlines_airline_the_to   
4       3   2451                     3_and_the_food_wine   
5       4   2419                       4_the_to_cable_of   
6       5   2158            5_drug_patients_drugs_cancer   
7       6   2117                       6_mr_the_firm_sec   
8       7   1987                         7_the_in_and_of   
9       8   1983                      8_ford_car_auto_gm   
10      9   1763              9_gas_power_energy_million   
11     10   1706                      10_the_he_game_his   
12     11   1475                 11_china_chinese_the_in   
13     12   1392         12_president_named_officer_vice   
14     13   1303          13_students_school_schools_the   
15     14   1297            

In [21]:
fig1 = topic_model.visualize_topics(width=1000, height=1000)
fig1.show()

In [7]:
# 可视化主题-关键词关系图
# 这张图可以清晰地看到每个主题的关键词及其重要性
fig1 = topic_model.visualize_barchart(top_n_topics=12) # 可视化前12个最频繁的主题
fig1.show()

# 可视化主题间的相似度矩阵
# 帮助我们理解不同主题之间的关联性
fig2 = topic_model.visualize_heatmap(n_clusters=11, top_n_topics=12)
fig2.show()

In [8]:
def topic_differences(model, original_topics, nr_topics=5):
    """Show the differences in topic representations between two models """
    df = pd.DataFrame(columns=["Topic", "Original", "Updated"])
    for topic in range(nr_topics):

        # Extract top 5 words per topic per model
        og_words = " | ".join(list(zip(*original_topics[topic]))[0][:5])
        new_words = " | ".join(list(zip(*model.get_topic(topic)))[0][:5])
        df.loc[len(df)] = [topic, og_words, new_words]

    return df

In [9]:
from copy import deepcopy
original_topics = deepcopy(topic_model.topic_representations_)

In [12]:
prompt = """
I have a topic that contains the following documents:
[DOCUMENTS]

The topic is described by the following keywords: [KEYWORDS]

Based on the information above, extract a short topic label in the following format:
topic: <short topic label>
"""

# Update our topic representations using GPT-3.5
client = openai.OpenAI(api_key="YOUR_OPENAI_API_KEY")

representation_model = OpenAI(
    client, model='gpt-4o' ,exponential_backoff=True, chat=True, prompt=prompt
)

topic_model.update_topics(docs, representation_model=representation_model)

# Show topic differences
topic_differences(topic_model, original_topics)

100%|██████████| 58/58 [02:07<00:00,  2.20s/it]


,Topic,Original,Updated
0,0,the | to | in | of | and,U.S. Military Strategy and Diplomacy in Middle...
1,1,tax | the | to | mr | of,U.S. Tax Policy and Political Challenges
2,2,airlines | airline | the | to | of,U.S. Airline Industry Post-9/11 Economic Impac...
3,3,and | the | food | wine | of,Wine Recommendations and Culture
4,4,the | to | cable | of | and,AOL Time Warner Merger and Media Industry Chal...


In [20]:
# Save the model using safetensors
topic_model.save("model_dir", serialization="safetensors")

In [19]:
df_result = topic_model.get_topic_info()
df_result.to_csv('result_1.csv')